# GPU (Batched) Test for Fisher Market

Tests the batched workspace path that works on both CPU (dense) and GPU (CuArray).

- `device=identity`: CPU dense batched (no CUDA needed)
- `device=cu`: GPU (requires `using CUDA`)

In [ ]:
using Pkg
Pkg.activate("../")

In [ ]:
using Revise
using Random, SparseArrays, LinearAlgebra, Printf
using ExchangeMarket
include("./setup.jl")

In [ ]:
# Set device: :cpu for batched CPU, :gpu for CUDA
# using CUDA
device = :cpu
# device = :gpu
println("Device: $device")

## Setup

In [ ]:
Random.seed!(1)
n = 500
m = 1000
ρ = 0.5

f0 = FisherMarket(m, n; ρ=ρ, bool_unit=true, scale=30.0, sparsity=0.2)
linconstr = LinearConstr(1, n, ones(1, n), [sum(f0.w)])
p₀ = ones(n) * sum(f0.w) / n
println("n=$n, m=$m, ρ=$ρ")

## 1. Ground truth via per-agent CPU (LogBar)

In [ ]:
f_gpu = copy(f0)
f_gpu.x .= ones(n, m) ./ m
f_gpu.p .= p₀

to_device!(f_gpu, device)

alg_gpu = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
traj_gpu = opt!(alg_gpu, f_gpu; keep_traj=true)
validate(f_gpu, alg_gpu)

## 2. Batched workspace (CPU or GPU)

In [ ]:
f_gpu = copy(f0)
f_gpu.x .= ones(n, m) ./ m
f_gpu.p .= p₀

if device === identity
    f_gpu.workspace = cpu_workspace(f_gpu)
else
    to_device!(f_gpu; device=device)
end

alg_gpu = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
traj_gpu = opt!(alg_gpu, f_gpu; keep_traj=true)
validate(f_gpu, alg_gpu)

# Per-agent CPU timing
f1 = copy(f0); f1.x .= ones(n,m)./m; f1.p .= p₀
alg1 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
t_cpu = @elapsed opt!(alg1, f1; keep_traj=false)

# Batched timing
f2 = copy(f0); f2.x .= ones(n,m)./m; f2.p .= p₀
to_device!(f2, device)
alg2 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
t_gpu = @elapsed opt!(alg2, f2; keep_traj=false)

@printf("CPU per-agent: %.3fs\n", t_cpu)
@printf("Batched (%s):  %.3fs\n", device, t_gpu)
@printf("Speedup:       %.2fx\n", t_cpu / t_gpu)

In [ ]:
@printf("CPU per-agent:  gₙ=%.2e  iters=%d\n", alg_cpu.gₙ, length(traj_cpu))
@printf("Batched:        gₙ=%.2e  iters=%d\n", alg_gpu.gₙ, length(traj_gpu))
@printf("\n|p diff|    = %.2e\n", maximum(abs.(alg_cpu.p .- alg_gpu.p)))
@printf("|x diff|    = %.2e\n", maximum(abs.(f_cpu.x .- f_gpu.x)))
@printf("|val_u diff|= %.2e\n", maximum(abs.(f_cpu.val_u .- f_gpu.val_u)))

# PropRes per-agent
f3 = copy(f0); f3.x .= ones(n,m)./m; f3.p .= p₀
(_, method_pr, kwargs_pr) = method_kwargs[7]  # PropRes
alg3 = method_pr(n, m, copy(p₀); linconstr=linconstr, kwargs_pr...)
opt!(alg3, f3; keep_traj=false, pₛ=copy(alg_cpu.p), maxiter=200)
@printf("PropRes per-agent: gₙ=%.2e\n", alg3.gₙ)

# PropRes batched
f4 = copy(f0); f4.x .= ones(n,m)./m; f4.p .= p₀
to_device!(f4, device)
alg4 = method_pr(n, m, copy(p₀); linconstr=linconstr, kwargs_pr...)
opt!(alg4, f4; keep_traj=false, pₛ=copy(alg_cpu.p), maxiter=200)
@printf("PropRes batched:   gₙ=%.2e\n", alg4.gₙ)
@printf("|p diff| = %.2e\n", maximum(abs.(alg3.p .- alg4.p)))

In [ ]:
# Per-agent CPU timing
f1 = copy(f0); f1.x .= ones(n,m)./m; f1.p .= p₀
alg1 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
t_cpu = @elapsed opt!(alg1, f1; keep_traj=false)

# Batched timing
f2 = copy(f0); f2.x .= ones(n,m)./m; f2.p .= p₀
if device === identity
    f2.workspace = cpu_workspace(f2)
else
    to_device!(f2; device=device)
end
alg2 = method(n, m, copy(p₀); linconstr=linconstr, kwargs...)
t_gpu = @elapsed opt!(alg2, f2; keep_traj=false)

@printf("CPU per-agent: %.3fs\n", t_cpu)
@printf("Batched:       %.3fs\n", t_gpu)
@printf("Speedup:       %.2fx\n", t_cpu / t_gpu)

for ρ_test in [0.5, -0.9, 0.9]
    f_t = FisherMarket(m, n; ρ=ρ_test, bool_unit=true, scale=30.0, sparsity=0.2)
    lc = LinearConstr(1, n, ones(1, n), [sum(f_t.w)])
    p0 = ones(n) * sum(f_t.w) / n

    # per-agent
    fa = copy(f_t); fa.x .= ones(n,m)./m; fa.p .= p0
    aa = method_kwargs[1][2](n, m, copy(p0); linconstr=lc, method_kwargs[1][3]...)
    opt!(aa, fa; keep_traj=false)

    # batched
    fb = copy(f_t); fb.x .= ones(n,m)./m; fb.p .= p0
    to_device!(fb, device)
    ab = method_kwargs[1][2](n, m, copy(p0); linconstr=lc, method_kwargs[1][3]...)
    opt!(ab, fb; keep_traj=false)

    err = maximum(abs.(aa.p .- ab.p))
    @printf("ρ=%+.1f  agent gₙ=%.1e  batch gₙ=%.1e  |p diff|=%.1e  %s\n",
        ρ_test, aa.gₙ, ab.gₙ, err, err < 1e-8 ? "✓" : "✗")
end

In [ ]:
# PropRes per-agent
f3 = copy(f0); f3.x .= ones(n,m)./m; f3.p .= p₀
(_, method_pr, kwargs_pr) = method_kwargs[7]  # PropRes
alg3 = method_pr(n, m, copy(p₀); linconstr=linconstr, kwargs_pr...)
opt!(alg3, f3; keep_traj=false, pₛ=copy(alg_cpu.p), maxiter=200)
@printf("PropRes per-agent: gₙ=%.2e\n", alg3.gₙ)

# PropRes batched
f4 = copy(f0); f4.x .= ones(n,m)./m; f4.p .= p₀
if device === identity
    f4.workspace = cpu_workspace(f4)
else
    to_device!(f4; device=device)
end
alg4 = method_pr(n, m, copy(p₀); linconstr=linconstr, kwargs_pr...)
opt!(alg4, f4; keep_traj=false, pₛ=copy(alg_cpu.p), maxiter=200)
@printf("PropRes batched:   gₙ=%.2e\n", alg4.gₙ)
@printf("|p diff| = %.2e\n", maximum(abs.(alg3.p .- alg4.p)))

## 6. Multiple ρ values

In [ ]:
for ρ_test in [0.5, -0.9, 0.9]
    f_t = FisherMarket(m, n; ρ=ρ_test, bool_unit=true, scale=30.0, sparsity=0.2)
    lc = LinearConstr(1, n, ones(1, n), [sum(f_t.w)])
    p0 = ones(n) * sum(f_t.w) / n

    # per-agent
    fa = copy(f_t); fa.x .= ones(n,m)./m; fa.p .= p0
    aa = method_kwargs[1][2](n, m, copy(p0); linconstr=lc, method_kwargs[1][3]...)
    opt!(aa, fa; keep_traj=false)

    # batched
    fb = copy(f_t); fb.x .= ones(n,m)./m; fb.p .= p0
    if device === identity
        fb.workspace = cpu_workspace(fb)
    else
        to_device!(fb; device=device)
    end
    ab = method_kwargs[1][2](n, m, copy(p0); linconstr=lc, method_kwargs[1][3]...)
    opt!(ab, fb; keep_traj=false)

    err = maximum(abs.(aa.p .- ab.p))
    @printf("ρ=%+.1f  agent gₙ=%.1e  batch gₙ=%.1e  |p diff|=%.1e  %s\n",
        ρ_test, aa.gₙ, ab.gₙ, err, err < 1e-8 ? "✓" : "✗")
end